# 0. 숫자 음성 합성 (edge-tts)

`data/digits/normal/`에 숫자 음성 60개(목소리 6개 × zero~nine)를 만듭니다.

- 저장소에는 실험에 실제로 사용한 60개 파일이 이미 들어 있으므로 **이 노트북은 다시 실행하지 않아도 됩니다.** 이미 있는 파일은 건너뜁니다.
- edge-tts는 Microsoft Edge의 온라인 신경망 TTS를 사용하므로 인터넷이 필요하며, 서비스가 바뀌면 같은 음성이 다시 만들어진다는 보장이 없습니다.
- 각 숫자 음성은 앞뒤 무음을 자르고(top_db=40) RMS를 0.05로 맞춥니다(논문 3.3절).
- Jupyter에서 실행해야 합니다(셀에서 `await`를 바로 사용).

In [ ]:
# ===== 0-1. 설정 =====
from pathlib import Path
import numpy as np
import soundfile as sf
import librosa
import edge_tts
from faster_whisper import decode_audio

ROOT = Path.cwd() # 저장소 최상위 폴더에서 실행
FS = 16000 # 샘플링 주파수 (Whisper 기준)
DIGIT_WORDS = ["zero", "one", "two", "three", "four",
               "five", "six", "seven", "eight", "nine"]
VOICES = [
    "en-US-GuyNeural", "en-US-ChristopherNeural", "en-GB-RyanNeural", # 남성
    "en-US-JennyNeural", "en-US-AriaNeural", "en-GB-SoniaNeural", # 여성
]
RATES = {"normal": "+0%"} # 말 속도
BANK = ROOT / "data/digits"
BANK.mkdir(parents=True, exist_ok=True)

In [ ]:
# ===== 0-2. 목소리 확인 =====
voices = await edge_tts.list_voices()
names = {v["ShortName"] for v in voices}
missing = [v for v in VOICES if v not in names]
assert not missing, f"edge-tts에서 찾을 수 없는 목소리: {missing}"
print("목소리", len(VOICES), "개 확인 완료")

In [ ]:
# ===== 0-3. 숫자 음성 합성 =====
async def tts_to_array(text, voice, rate):
    tmp = BANK / "_tmp.mp3"
    await edge_tts.Communicate(text, voice, rate=rate).save(str(tmp)) # 음성 합성
    audio = decode_audio(str(tmp), sampling_rate=FS) # 16kHz로 읽기
    audio, _ = librosa.effects.trim(audio, top_db=40) # 앞뒤 무음 자르기
    audio = audio / np.sqrt(np.mean(audio**2)) * 0.05 # 음량 통일 (RMS 0.05)
    return audio

for rate_name, rate in RATES.items():
    folder = BANK / rate_name
    folder.mkdir(exist_ok=True)
    for voice in VOICES:
        for d, word in enumerate(DIGIT_WORDS):
            out = folder / f"{voice}_{d}.wav"
            if out.exists():
                continue
            audio = await tts_to_array(word, voice, rate)
            sf.write(out, audio, FS)
        print(rate_name, voice, "완료")
print("숫자 음성:", len(list((BANK / "normal").glob("*.wav"))), "개")